# MACS 30111 — M7 Pandas, days 1–2: in-class notebook
Slides: `M7_Pandas_30111.pptx`, slides 1–52 (day 2 starts at section 7) · Textbook: §4.3.1–4.3.6 · Docs: https://pandas.pydata.org/docs/

**How to use this notebook:** run the **Example** cells as we go. In each **Try it** cell, predict the answer first, then write the code.

**Data:** the 2015 NYC Street Tree Census (683,788 trees). Put the CSV in a folder called `data/` next to this notebook.

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 12)   # wide tables: show 12 columns, '...' for the rest

## 1. Series and DataFrames (slides 8–13)
A **Series** is one labeled column: values plus an **index** (the labels). A **DataFrame** is a table: several Series that share the same index.

### Example: build a Series (slide 10)

In [ ]:
# from a list, plus a list of labels
s = pd.Series([90, 95, 90], index=["Rory", "Lorelai", "Luke"])
s

In [ ]:
# from a dictionary: the keys become the index
s = pd.Series({"Rory": 90, "Lorelai": 95, "Luke": 90})
s

### Example: build a DataFrame (slides 11–13)

In [ ]:
# dictionary of lists: the keys become the COLUMNS
data = {"PA1": [90, 95, 90], "PA2": [88, 90, 75]}
df = pd.DataFrame(data, index=["Rory", "Lorelai", "Luke"])
df

In [ ]:
# dictionary of dictionaries: outer keys -> columns, inner keys -> rows. Same table!
data = {"PA1": {"Rory": 90, "Lorelai": 95, "Luke": 90},
        "PA2": {"Rory": 88, "Lorelai": 90, "Luke": 75}}
pd.DataFrame(data)

In [ ]:
# from a NumPy array: you supply the row and column labels
rng = np.random.default_rng(13)
arr = rng.uniform(0, 1, (3, 2))
pd.DataFrame(arr, index=["Rory", "Lorelai", "Luke"], columns=["PA1", "PA2"])

In [ ]:
# first look: what's in it?
print(df.shape)    # (rows, columns)
print(df.dtypes)   # one type per column
df["PA1"]          # one column -> a Series

### Try it 1.1
Make a DataFrame called `scores` for four students, **Dwight**, **Angela**, **Pam** and **Jim**, with columns `SE1` = 8, 10, 9, 7 and `SE2` = 7, 9, 10, 8. Then:
1. Print its shape.
2. Get the `SE2` column. What type is it?
3. Compute the class average on `SE1` (`.mean()` works on a Series, just like NumPy).

In [ ]:
# your code here


## 2. Load real data (slides 14–15, 20)
`pd.read_csv` reads a file into a DataFrame. This one is big, so it takes a few seconds.

**Questions we'll be able to answer by the end of the module (slide 20):**
1. How many different species are planted as street trees in New York?
2. What are the five most common street tree species?
3. What is the most common street tree species in Brooklyn?
4. What percentage of the street trees in Queens are dead or in poor health? **← today**
5. How does street tree health differ by borough?

### Example: load and look

In [3]:
trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv")
trees.shape

(100000, 45)

In [ ]:
trees.head()

In [ ]:
trees.dtypes.head(10)   # 'object' = text

### Try it 2.1
1. Show the **last 3** rows.
2. How many columns are there? (Use `.shape`, not counting by hand.)
3. Print the column names. Which column holds the borough?

In [ ]:
# your code here


## 3. Missing data (slide 19)
`.isna()` gives True where a value is missing (`NaN`). Because True counts as 1, `.isna().sum()` counts the missing values in each column.

In [ ]:
trees.isna().sum().sort_values(ascending=False).head(8)

### Try it 3.1
**Predict first:** why would a tree have no `health` value?

Then check: how many trees have a missing `health`, and what is their `status`? *(Hint: filter to the trees with missing health, then look at the `status` column; `.value_counts()` counts each value.)*

In [ ]:
# your code here


## 4. Selecting rows and columns (slides 21–24)
- `iloc[...]` selects by **position**: row 0, 1, 2, ...
- `loc[...]` selects by **label**: whatever is in the index.

### Example: predict, then run
`trees.head()` shows that the first row is tree **180683**. **Predict:** do these two give the same tree?

In [ ]:
trees.iloc[0][["tree_id", "borough", "spc_common"]]

In [ ]:
trees.loc[180683][["tree_id", "borough", "spc_common"]]

They don't. Right now the index is just row numbers 0, 1, 2, ..., so `loc[180683]` means "the row **labeled** 180683", and our 100,000-tree sample only has labels 0 to 99,999, so it raises a `KeyError`. (In the full 683,788-tree file it would quietly return row 180,683, the wrong tree.)

### Example: make the tree id the index
Now the labels *are* tree ids, so `loc` finds the tree by its id.

In [ ]:
trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv", index_col="tree_id")
trees.loc[180683][["borough", "spc_common", "health"]]

### Example: one column vs. a list of columns (slide 23)
**Predict** the type of each before running.

In [ ]:
print(type(trees["health"]))      # one name  -> a Series
print(type(trees[["health"]]))    # a list    -> a DataFrame
trees[["health", "borough"]].head()

In [ ]:
# rows and columns together (slide 24)
print(trees.iloc[:5][["health", "borough"]])                     # first 5 rows, by position
trees.loc[[180683, 193310, 203805], ["health", "borough"]]       # three trees, by id

### Try it 4.1
1. Get the tree in the **10th row** (careful: positions start at 0).
2. Get tree **203805** (the tree id). What species is it, and what borough?
3. Get the `spc_common` and `tree_dbh` columns for the first 5 trees.

In [ ]:
# your code here


## 5. Filtering with Boolean masks (slide 25)
Same idea as NumPy: a comparison gives True/False for every row, and `trees[mask]` keeps the True rows. Combine conditions with `&` (and) and `|` (or), with **parentheses around each one**.

Name your mask `mask` or something descriptive, not `filter` (that's a built-in Python function).

In [ ]:
is_dead = trees["status"] == "Dead"
print(is_dead.head())        # one True/False per tree
print(is_dead.sum())         # 13,961 dead trees
trees[is_dead][["status", "health", "borough"]].head()

In [ ]:
# the opposite: keep everything that is NOT dead
trees[trees["status"] != "Dead"].shape

### Try it 5.1: motivating question 4
**What percentage of the street trees in Queens are dead or in poor health?**

Steps: (1) keep only the Queens trees; (2) build a mask for "dead **or** poor"; (3) divide.

*Think about it:* dead trees have `health = NaN`. What would you miss if you only checked `health == "Poor"`?

In [ ]:
# your code here


### Try it 5.2
How many **red maples** in **Brooklyn** are in **Good** health? *(Three conditions, all with `&`.)*

In [ ]:
# your code here


## 6. Challenge (if you finish early): motivating questions 1–3
You'll need two new tools: `.nunique()` (how many different values) and `.value_counts()` (how many of each value, largest first).

In [ ]:
# 1. How many different species?


In [ ]:
# 2. The five most common species


In [ ]:
# 3. The most common species in Brooklyn


# Day 2: describe, apply, group, reshape
Slides 26–52 · Textbook §4.3.5–4.3.6

**Today's question (slide 29): how does tree health vary by borough?** Which columns would you need? What tools do you have so far?

## 7. Review: load with row labels (slides 27–28)
Loading with `index_col="tree_id"` makes each tree's id its row label, so `loc` finds a tree by its id.

In [ ]:
trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv", index_col="tree_id")
trees.shape

### Try it 7.1
1. Get the tree in the **10th row**.
2. Get tree **180683** by its id. What species is it, and what borough?

In [ ]:
# your code here


### Try it 7.2 (on your own)
Get the **last** tree in the file by position. Then look up the same tree again by its id with `loc`. What species and borough is it?

In [ ]:
# your code here


## 8. Summary statistics and sorting (slides 30–33)
**Predict first:** `trees.count()` counts the non-missing values in each column. Will `health` have the same count as `status`? Why or why not?

In [ ]:
trees.count()[["status", "health", "spc_common", "tree_dbh"]]

In [ ]:
trees["status"].value_counts()

In [ ]:
trees["tree_dbh"].describe()

### Try it 8.1
1. Why is `health`'s count lower than `status`'s? *(Hint: compare the missing count with the `value_counts` above.)*
2. Show the 3 widest trees (`tree_dbh`, largest first) with their species and borough. Does the biggest one look real?

In [4]:
# your code here
trees.sort_values(by="tree_dbh")

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,...,x_sp,y_sp,council district,census tract,bin,bbl
13530,330265,510073,10/15/2015,0,5,OnCurb,...,1029944.915,244170.9034,13.0,194.0,2073556.0,2.053500e+09
28961,400951,330508,10/29/2015,0,30,OnCurb,...,1056016.902,177086.3040,31.0,664.0,4292063.0,4.137690e+09
70812,669766,319633,08/18/2016,0,13,OnCurb,...,1050981.050,206148.0796,23.0,129102.0,4164628.0,4.077850e+09
70838,689990,214808,08/31/2016,0,12,OnCurb,...,1008352.587,172386.9758,NaN,NaN,NaN,NaN
70843,692662,214673,09/01/2016,0,22,OnCurb,...,1011582.037,172350.3693,46.0,990.0,3230875.0,3.082200e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35562,426541,202420,11/04/2015,142,0,OnCurb,...,1021795.854,187199.9990,37.0,1188.0,3094115.0,3.041840e+09
49607,557343,311449,05/09/2016,181,0,OnCurb,...,1034641.101,213133.7765,20.0,1205.0,4117384.0,4.052000e+09
30013,363589,512207,10/22/2015,204,0,OnCurb,...,1029007.500,252190.7506,13.0,310.0,2050922.0,2.043970e+09
55271,567108,214010,05/26/2016,248,0,OnCurb,...,1003317.140,169586.7875,45.0,726.0,3214354.0,3.077500e+09


### Try it 8.2 (on your own)
1. What are the mean and median `tree_dbh` of the **living** trees only?
2. How many trees have `tree_dbh` equal to 0? What is their `status`? Why does that make sense?

In [ ]:
# your code here


## 9. Applying functions (slides 34–35)
- `Series.apply(f)` calls `f` on each value.
- `DataFrame.apply(f)` calls `f` on each **column** (a Series) by default; `axis=1` does each row.
- `DataFrame.map(f)` calls `f` on every single value. *(The slide's older name, `applymap`, was removed in pandas 3.)*

**Predict:** what is the shape of `df`, and what does `df.apply(from_celsius)` give?

In [ ]:
def from_celsius(temp):
    return temp * 9 / 5 + 32

df = pd.DataFrame([list(range(0, 100, 10)), list(range(5, 105, 10))])
print(df.shape)
df.apply(from_celsius)

In [ ]:
# same result, one value at a time (this used to be called applymap)
df.map(from_celsius)

### Try it 9.1
`tree_dbh` is in inches. Make a new column `dbh_cm` with the diameter in centimeters (1 inch = 2.54 cm) using `apply`. Then do it again **without** `apply`. Which is simpler?

In [ ]:
# your code here


### Try it 9.2 (on your own)
Write a function `size_class(dbh)` that returns `"small"` for a diameter under 6 inches, `"medium"` for 6 up to (not including) 18, and `"large"` otherwise. Use `apply` to make a `size` column, then count the living trees in each size class.

In [ ]:
# your code here


## 10. Grouping and unstacking (slides 36–40, textbook §4.3.5)
`groupby` **splits** the rows into groups, **applies** a function to each group, and **combines** the results.

**Predict:** which borough has the widest trees on average?

In [ ]:
trees.groupby("borough")[["tree_dbh"]].mean()

In [ ]:
# two columns: one row per (borough, status) pair
counts = trees.groupby(["borough", "status"]).size()
counts

In [ ]:
# unstack moves the inner index level (status) into the columns
counts.unstack()

### Try it 10.1
1. Before running: how many rows does `counts` have, and what is its index?
2. Make a table with one row per borough and one column per **health** value (Fair, Good, Poor), holding the number of trees.

In [ ]:
# your code here


### Try it 10.2 (on your own)
1. How many **different species** are planted in each borough? *(Hint: `.nunique()` works after `groupby` too.)*
2. Make a table of the **average** `tree_dbh` for each borough (rows) and status (columns). Why is every Stump value 0?

In [ ]:
# your code here


## 11. Answer the question: health by borough (slide 44)
With a partner, write a comment on each line of the function before you run it. What does `.where` do on the first line?

In [5]:
def tree_health_by_boro(trees):
    combined_status = trees.status.where(trees.status != "Alive", trees.health)
    num_per_boro = trees.groupby("borough").size()
    combined_per_boro = trees.groupby(["borough", combined_status]).size()
    pct_per_boro = combined_per_boro / num_per_boro * 100.0
    pct_per_boro_df = pct_per_boro.unstack()
    return pct_per_boro_df[["Good", "Fair", "Poor", "Dead", "Stump"]]

tree_health_by_boro(trees).round(1)

status,Good,Fair,Poor,Dead,Stump
borough,,,,,
Bronx,78.2,13.1,3.5,3.0,2.2
Brooklyn,77.8,14.3,3.6,1.9,2.4
Manhattan,72.5,17.5,5.2,2.9,1.9
Queens,77.6,13.9,3.6,1.7,3.2
Staten Island,77.6,14.3,4.2,1.9,1.9


### Try it 11.1
Which borough has the lowest share of trees in **Good** health? Which has the highest share in **Poor** health? Answer with code, not by eye.

In [ ]:
# your code here


### Try it 11.2 (on your own)
Check one number from the function's table **without** the function: what percentage of all Brooklyn trees are alive and in Good health? *(Filter to Brooklyn, then count.)*

In [ ]:
# your code here


## 12. Tidy data and pivoting (slides 45–51, textbook "Pivoting")
**Tidy data:** each variable is a column, each observation is a row. *Long* tables are easy to analyze; *wide* tables are easy to read. `pivot` turns long into wide: you name a column for the **index**, a column for the new **columns**, and a column for the **values**.

Start from the percentage of trees with each status in each borough (a Series with a two-level index):

In [ ]:
pct_per_boro = (trees.groupby(["borough", "status"]).size()
                / trees.groupby("borough").size() * 100)
pct_per_boro.head()

In [ ]:
# Predict: will this work? (Quiz question 6!)
pct_per_boro.pivot(index="borough", columns="status")

It raises an error: `pct_per_boro` is a **Series**, and borough and status are in its **index**, not in columns, so there is nothing for `pivot` to use. `reset_index` moves the index into ordinary columns:

In [ ]:
long = pct_per_boro.to_frame().reset_index()
long = long.rename(columns={0: "percentage"})
long.head()

In [ ]:
wide = long.pivot(index="borough", columns="status", values="percentage")
wide.round(1)

### Try it 12.1
Build the **count** table from section 10 again, but with `pivot` instead of `unstack`:
`groupby` → `size()` → `reset_index(name="n")` → `pivot`. Check that the shapes of the long and wide tables make sense.

In [ ]:
# your code here


### Try it 12.2 (on your own)
1. Build a wide table of **health** counts with `pivot`: one row per borough, one column per health value (Fair, Good, Poor).
2. Add a column `pct_poor`: the percentage of each borough's rated trees that are Poor. Which borough is highest?

In [ ]:
# your code here


## Next time
Merging a second dataset (slides 53–58) and plotting. Homework: the **on your own** Try its (7.2–12.2) and textbook §4.3.5–4.3.6.